In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
train_df = pd.read_csv("../input/house-prices-advanced-regression-techniques/train.csv")
test_df = pd.read_csv("../input/house-prices-advanced-regression-techniques/test.csv")
sample_df = pd.read_csv("../input/house-prices-advanced-regression-techniques/sample_submission.csv")

In [ ]:
train_df

In [ ]:
test_df

In [ ]:
sample_df

## 文字列（カテゴリ変数）の対応

In [ ]:
X = train_df[["MSZoning", "LotArea"]]
# ここで対数を取る
y = np.log1p(train_df["SalePrice"])
X_test = test_df[["MSZoning", "LotArea"]]

In [ ]:
X

In [ ]:
from sklearn.model_selection import train_test_split

注：説明のわかりやすさのためにここに書いているが、↑のようなimport系は本当は一番上にまとめるのがお行儀の良い書き方

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, random_state=0)

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
model = LinearRegression()

In [ ]:
# これをやるとエラー
# ValueError: could not convert string to float: 'RM'
# model.fit(X_train, y_train)

### One hot Encoding

In [ ]:
X_2 = pd.get_dummies(X)
X_test_2 = pd.get_dummies(X_test)


In [ ]:
X_2

In [ ]:
X_test_2

学習とテストでカテゴリが食い違う場合は対処が必要なので注意

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(X_2, y, random_state=0)

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

True = 1, False = 0として数値とみなすことができている

## 欠損の対応

In [ ]:
X = train_df[["LotFrontage", "LotArea"]]
# ここで対数を取る
y = np.log1p(train_df["SalePrice"])
X_test = test_df[["LotFrontage", "LotArea"]]

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, random_state=0)

In [ ]:
# これをやるとエラー
# ValueError: Input X contains NaN.
model = LinearRegression()
# model.fit(X_train, y_train)

#### 対処1. 0で埋める
考えた結果負の数とか大きい数で埋める場合はベタ書きしている値を書き換える

In [ ]:
X["LotFrontage"] = X["LotFrontage"].fillna(0)
X_test["LotFrontage"] = X_test["LotFrontage"].fillna(0)

↑怒られるけどあまり気にしないでOK

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, random_state=0)

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

#### 対処2. 定数で埋める

In [ ]:
X = train_df[["LotFrontage", "LotArea"]]
# ここで対数を取る
y = np.log1p(train_df["SalePrice"])
X_test = test_df[["LotFrontage", "LotArea"]]

In [ ]:
X["LotFrontage"] = X["LotFrontage"].fillna(X["LotFrontage"].mean())
X_test["LotFrontage"] = X_test["LotFrontage"].fillna(X["LotFrontage"].mean())

X_test側も同じ数値で埋めていることに注目  
これは「他のテストデータに何が入っているか」は出力に影響すべきでないという思想からきている

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, random_state=0)

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

#### 対処3. 欠損フラグを残した上で0で埋める 
欠損を何で埋めるかを機械学習に任せるのと同じ、精度は出る代わりに過学習のリスクアップと、説明性の低下

In [ ]:
X = train_df[["LotFrontage", "LotArea"]]
# ここで対数を取る
y = np.log1p(train_df["SalePrice"])
X_test = test_df[["LotFrontage", "LotArea"]]

In [ ]:
# 順番前後すると欠損がなくなるので注意
X["LotFrontage_na"] = X["LotFrontage"].isna()
X["LotFrontage"] = X["LotFrontage"].fillna(0)
X_test["LotFrontage_na"] = X_test["LotFrontage"].isna()
X_test["LotFrontage"] = X_test["LotFrontage"].fillna(0)

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, random_state=0)

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

## 外れ値の対処

In [ ]:
X = train_df[["YearBuilt", "LotArea"]]
# ここで対数を取る
y = np.log1p(train_df["SalePrice"])
X_test = test_df[["YearBuilt", "LotArea"]]

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
plt.hist(X["YearBuilt"])

例えば1900年より前はおかしいと思ったとする（これが正しいかどうかは不明）

#### 丸め処理 

In [ ]:
X["YearBuilt"] = np.maximum(X["YearBuilt"], 1900)
X_test["YearBuilt"] = np.maximum(X_test["YearBuilt"], 1900)

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, random_state=0)

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

#### 学習データから除外
テストデータを除外することはできないので注意

In [ ]:
X = train_df[["YearBuilt", "LotArea"]]
# ここで対数を取る
y = np.log1p(train_df["SalePrice"])
X_test = test_df[["YearBuilt", "LotArea"]]

In [ ]:
X.shape

In [ ]:
y = y[X["YearBuilt"] >= 1900]
X = X[X["YearBuilt"] >= 1900]

In [ ]:
X.shape

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, random_state=0)

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)